# relatedb-ideas as a graph

Renders the markdown vault (`Project`, `Research`, `Technology`, `Company`,
`Person`, `Category`) with NVL via `relate.viz`, clustered by label.
Opens the vault **read-only** so it never takes the writer lock.
Exports `ideas-graph.svg` and `ideas-graph.png` next to the vault.

In [1]:
import relate
from neo4j_viz import (Node as VizNode, Relationship as VizRelationship, VisualizationGraph,
                       ClusteringOptions, ForceDirectedLayoutOptions, Layout)
from neo4j_viz.colors import NEO4J_COLORS_DISCRETE, ColorSpace

VAULT = "/Users/mh/d/llm/relatedb-ideas"
db = relate.connect(VAULT, read_only=True)
deg = {r["id"]: r["d"] for r in db.execute(
    "MATCH (n) OPTIONAL MATCH (n)-[r]-() RETURN elementId(n) AS id, count(r) AS d")}
nodes = [VizNode(id=r["id"], caption=r["caption"], size=5 + 2.0 * (deg.get(r["id"], 0) ** 0.5),
                 properties={"labels": [r["label"]], "group": r["label"]})
         for r in db.execute("MATCH (n) RETURN elementId(n) AS id, labels(n)[0] AS label, coalesce(n.name, elementId(n)) AS caption")]
rels = [VizRelationship(id=r["id"], source=r["src"], target=r["tgt"], caption=r["t"], properties={"type": r["t"]})
        for r in db.execute("MATCH (a)-[r]->(b) RETURN elementId(r) AS id, elementId(a) AS src, elementId(b) AS tgt, type(r) AS t")]
vg = VisualizationGraph(nodes=nodes, relationships=rels)
vg.color_nodes(property="labels", color_space=ColorSpace.DISCRETE, colors=NEO4J_COLORS_DISCRETE)
widget = vg.render_widget(layout=Layout.FORCE_DIRECTED, theme="light",
                          width="2000px", height="1400px", show_layout_button=False, show_search_button=False,
                          layout_options=ForceDirectedLayoutOptions(
                              clustering=ClusteringOptions(clusterBy="group", drawOutline=True, drawHalo=True)))
widget

/Users/mh/d/rust/relate/surfaces/relate-python/.venv/lib/python3.13/site-packages/neo4j_viz/options.py:164: UserWarning: The force-directed layout options ['clustering'] have no effect on `Layout.FORCE_DIRECTED` (NVL's d3-force layout has no tunable options); use `Layout.GPU_FORCE` to apply them.
  warnings.warn(


In [2]:
from pathlib import Path
out = Path(VAULT)
await widget.save(out / "ideas-graph.svg", format="svg", timeout=180)
await widget.save(out / "ideas-graph.png", format="png", timeout=180)
print("saved", (out / "ideas-graph.svg").stat().st_size, (out / "ideas-graph.png").stat().st_size)

saved 1312513 3918378
